#utils/data_preparation/helpers.py
(Módulo de funciones auxiliares, no es una clase pero necesario)

In [0]:
# helpers.py
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

def add_codmes_spark(col_name, months):
    """
    Suma meses a un código de mes en formato 'yyyyMM' y devuelve entero.
    Ejemplo: add_codmes_spark('202501', +1) -> 202502
    """
    return F.expr(f"""
        cast(
            date_format(
                add_months(
                    to_date(cast({col_name} as string), 'yyyyMM'),
                    {months}
                ),
                'yyyyMM'
            ) as int
        )
    """)

def _operaciones_max_between_cols(columnas):
    """Función interna para calcular el máximo entre valores numéricos, ignorando nulos."""
    valor_inicial = float(-99999999999999.0)
    mayor = float(0.0)
    for col in columnas:
        if col is not None and str(col) != "" and str(col).upper() != "NULL":
            numero = float(col)
        else:
            numero = valor_inicial
        if numero >= mayor:
            mayor = numero
    return None if mayor == valor_inicial else mayor

operaciones_max_between_cols_udf = udf(_operaciones_max_between_cols, DoubleType())

#utils/data_preparation/universe_builder.py

In [0]:
# universe_builder.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger

class UniverseBuilder:
    """
    Construye el universo de clientes PYME activos (portafolio) para un mes de proceso.
    Aplica filtros: productos PYME revolvente/no revolvente, cliente no nulo, antigüedad >0.
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, source_table: str, mes_proceso: int) -> DataFrame:
        self.logger.info(f"Construyendo universo para mes {mes_proceso} desde {source_table}")
        df = self.spark.table(source_table).select(
            "codmes", "codclaveunicocli", "codinternocomputacional", "codclavepartycli"
        ).filter(
            F.trim(F.col("codproductonivel0rbm")).isin("PYME REVOLVENTE", "PYME NO REVOLVENTE")
            & (F.col("codmes") == mes_proceso)
            & F.col("codclaveunicocli").isNotNull()
            & (F.col("ctdmesmaduracion") > 0)
        ).distinct()
        universe = df.groupBy("codclaveunicocli", "codmes").agg(
            F.max("codinternocomputacional").alias("codinternocomputacional"),
            F.max("codclavepartycli").alias("codclavepartycli")
        )
        count = universe.count()
        self.logger.info(f"Universo generado: {count} registros")
        return universe

#utils/data_preparation/relationship_builder.py

In [0]:
# relationship_builder.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger
from utils.data_preparation.helpers import add_codmes_spark

class RelationshipBuilder:
    """
    Identifica la relación de dueño (DUENIO) para cada cliente.
    Lógica: selecciona el dueño principal (si PJ, se prefiere PN si existe).
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, mes_data: int) -> DataFrame:
        self.logger.info(f"Construyendo relaciones de dueño con mes_data={mes_data}")
        cliente_prospecto = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rmbmcaym_modelogestion_vu.hm_clienteprospectopyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            "tippartyidentificacion"
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        relacionados = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_relacionadoapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            "CODCLAVEUNICOCLIREL",
            "DESTIPREL"
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        relacionados = relacionados.join(cliente_prospecto, ["codmes", "CODCLAVEUNICOCLI"], "left_outer")
        duenios = relacionados.filter(F.col("DESTIPREL").isin("DUENIO", "DUENIO P.N.")).select(
            "codmes", "CODCLAVEUNICOCLI", "CODCLAVEUNICOCLIREL",
            F.when(F.col("tippartyidentificacion") == "6", "J").otherwise("N").alias("FLGTIPPER")
        )
        duenio_unico = duenios.groupBy("codmes", "CODCLAVEUNICOCLI").agg(
            F.max(F.when(F.col("FLGTIPPER") == "N", F.col("CODCLAVEUNICOCLI"))
                  .otherwise(F.col("CODCLAVEUNICOCLIREL"))).alias("CODCLAVEUNICOCLIREL")
        )
        relacionados_con_flag = relacionados.alias("A").join(
            duenio_unico.alias("B"),
            (F.col("A.CODCLAVEUNICOCLI") == F.col("B.CODCLAVEUNICOCLI")) &
            (F.col("A.CODCLAVEUNICOCLIREL") == F.col("B.CODCLAVEUNICOCLIREL")) &
            (F.col("A.codmes") == F.col("B.codmes")),
            "left_outer"
        ).select(
            F.col("A.codmes"), F.col("A.CODCLAVEUNICOCLI"), F.col("A.CODCLAVEUNICOCLIREL"),
            F.when(F.col("A.DESTIPREL").isin("DUENIO", "DUENIO P.N."),
                   F.when(F.col("B.CODCLAVEUNICOCLI").isNull(), 0).otherwise(1)
            ).otherwise(0).alias("FLGRELDUENIO")
        )
        return relacionados_con_flag.filter(F.col("FLGRELDUENIO") == 1).select(
            "codmes", "CODCLAVEUNICOCLI", "CODCLAVEUNICOCLIREL"
        )

#utils/data_preparation/demographic_builder.py

In [0]:
# demographic_builder.py
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import IntegerType
from utils.logger import MLOpsLogger
from utils.data_preparation.helpers import add_codmes_spark

class DemographicBuilder:
    """Obtiene edad y actividad económica del cliente a partir de la tabla de aprobación."""
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, mes_data: int) -> DataFrame:
        self.logger.info(f"Construyendo variables demográficas con mes_data={mes_data}")
        df = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_clienteaprobacionapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            F.col("NUMEDAD").cast(IntegerType()).alias("EDAD_FIN"),
            F.col("DESECCIONECONOMICAAPPPYME").alias("ACT_ECO_FIN"),
            "TIPPARTYIDENTIFICACION"
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")
        return df

#utils/data_preparation/carretera_builder.py

In [0]:
# carretera_builder.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger
from utils.data_preparation.helpers import add_codmes_spark

class CarreteraBuilder:
    """
    Construye variables base (carretera) a partir de matrices RCC, resumen de saldos y materialidad.
    Aplica lógica de materialidad: si existe columna '_100', se usa; si no, la base.
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, mes_data: int) -> DataFrame:
        self.logger.info(f"Construyendo carretera con mes_data={mes_data}")

        # RCC base
        rcc = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"), "codclaveunicocli",
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12").alias("ATRASOMAX_CRNENR_12"),
            F.col("RCC_MTO_ADE_ACT_SHIP_RT_U6M").alias("MONTOADE_ACT_MAX6_S_HIP"),
            F.col("RCC_PCT_UTL_3_RT_U3M").alias("UTL_3"),
            F.col("RCC_CTD_SF_CAL_CPP_FRQ_0_U24").alias("SF_NUM_CAL_CPP24")
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        # Resumen saldo
        resumen = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"), "codclaveunicocli",
            F.col("PROD_MTO_SLD_PRM_TSAV_FRQ_100_U24").alias("MESES_PMSAVMF_24_100")
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        # Materialidad externa (ATRASOMAX y SF_NUM_CAL_CPP24 con umbral 100)
        materialidad = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablecrcmaterialidadapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"), "codclaveunicocli",
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12").alias("ATRASOMAX_CRNENR_12_100"),
            F.col("RCC_CTD_SF_CAL_CPP_FRQ_100_U24").alias("SF_NUM_CAL_CPP24_100"),
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_100_U6M").alias("MESES_ACTIVO_SF_BU_MA6_100")
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        # Join
        carretera = rcc.join(resumen, ["codmes", "codclaveunicocli"], "fullouter")
        carretera = carretera.join(materialidad, ["codmes", "codclaveunicocli"], "fullouter")

        # Aplicar materialidad (coalesce o when)
        carretera = carretera.select(
            "codmes", "codclaveunicocli",
            F.coalesce("ATRASOMAX_CRNENR_12_100", "ATRASOMAX_CRNENR_12").alias("ATRASOMAX_CRNENR_12"),
            F.coalesce("SF_NUM_CAL_CPP24_100", "SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24"),
            "MONTOADE_ACT_MAX6_S_HIP", "UTL_3", "MESES_PMSAVMF_24_100",
            "MESES_ACTIVO_SF_BU_MA6_100"
        )

        # Base para meses activos (umbral 0)
        meses_base = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"), "codclaveunicocli",
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_0_U6M").alias("MESES_ACTIVO_SF_BU_MA6_0_BASE")
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")

        carretera = carretera.join(meses_base, ["codmes", "codclaveunicocli"], "left_outer")
        carretera = carretera.withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0",
            F.coalesce("MESES_ACTIVO_SF_BU_MA6_100", "MESES_ACTIVO_SF_BU_MA6_0_BASE")
        ).drop("MESES_ACTIVO_SF_BU_MA6_100", "MESES_ACTIVO_SF_BU_MA6_0_BASE")
        return carretera

#utils/data_preparation/owner_variables_builder.py

In [0]:
# owner_variables_builder.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger

class OwnerVariablesBuilder:
    """Construye variables del dueño (sufijo _RL) a partir de la relación y la carretera."""
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, relacion_df: DataFrame, carretera_df: DataFrame) -> DataFrame:
        self.logger.info("Construyendo variables del dueño (_RL)")
        # Renombrar columna de cliente en carretera para join con CODCLAVEUNICOCLIREL
        carretera_renamed = carretera_df.select(
            "codmes",
            F.col("codclaveunicocli").alias("CODCLAVEUNICOCLIREL"),
            "ATRASOMAX_CRNENR_12", "MONTOADE_ACT_MAX6_S_HIP", "UTL_3",
            "SF_NUM_CAL_CPP24", "MESES_ACTIVO_SF_BU_MA6_0", "MESES_PMSAVMF_24_100"
        )
        owner_vars = relacion_df.join(
            carretera_renamed,
            ["CODCLAVEUNICOCLIREL", "codmes"],
            "left_outer"
        ).select(
            "codmes", "CODCLAVEUNICOCLI",
            F.col("ATRASOMAX_CRNENR_12").alias("ATRASOMAX_CRNENR_12_RL"),
            F.col("MONTOADE_ACT_MAX6_S_HIP").alias("MONTOADE_ACT_MAX6_S_HIP_RL"),
            F.col("UTL_3").alias("UTL_3_RL"),
            F.col("SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24_RL"),
            F.col("MESES_ACTIVO_SF_BU_MA6_0").alias("MESES_ACTIVO_SF_BU_MA6_0_RL"),
            F.col("MESES_PMSAVMF_24_100").alias("MESES_PMSAVMF_24_100_RL")
        ).dropDuplicates(["codmes", "CODCLAVEUNICOCLI"])
        return owner_vars

#utils/data_preparation/aggregated_variables_builder.py

In [0]:
# aggregated_variables_builder.py
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import array
from utils.logger import MLOpsLogger
from utils.data_preparation.helpers import operaciones_max_between_cols_udf

class AggregatedVariablesBuilder:
    """
    Calcula variables agregadas cliente/dueño (_AG) usando la UDF de máximo.
    Para PJ: max(cliente, dueño). Para PN: valor del cliente.
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, universo_df: DataFrame, demographic_df: DataFrame,
              owner_vars_df: DataFrame, carretera_df: DataFrame) -> DataFrame:
        self.logger.info("Construyendo variables agregadas (_AG)")
        # Variables del cliente (para AG)
        client_vars = carretera_df.select(
            "codmes", "codclaveunicocli", "SF_NUM_CAL_CPP24", "MESES_ACTIVO_SF_BU_MA6_0"
        )

        # Unir todo
        base = universo_df.join(demographic_df, ["codmes", "CODCLAVEUNICOCLI"], "left_outer") \
                         .join(owner_vars_df, ["codmes", "CODCLAVEUNICOCLI"], "left_outer") \
                         .join(client_vars, ["codmes", "codclaveunicocli"], "left_outer")

        # Calcular campos AG
        base = base.withColumn(
            "SF_NUM_CAL_CPP24_AG_raw",
            operaciones_max_between_cols_udf(array("SF_NUM_CAL_CPP24", "SF_NUM_CAL_CPP24_RL"))
        ).withColumn(
            "SF_NUM_CAL_CPP24_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != "6", F.col("SF_NUM_CAL_CPP24"))
             .otherwise(F.col("SF_NUM_CAL_CPP24_AG_raw")).cast(IntegerType())
        ).withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG_raw",
            operaciones_max_between_cols_udf(array("MESES_ACTIVO_SF_BU_MA6_0", "MESES_ACTIVO_SF_BU_MA6_0_RL"))
        ).withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != "6", F.col("MESES_ACTIVO_SF_BU_MA6_0"))
             .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_AG_raw")).cast(IntegerType())
        )
        return base

#utils/data_preparation/sunat_seniority_builder.py

In [0]:
# sunat_seniority_builder.py
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from utils.logger import MLOpsLogger
from utils.data_preparation.helpers import add_codmes_spark

class SunatSeniorityBuilder:
    """Obtiene antigüedad en meses según SUNAT (ctdmesantiguedadempsunat)."""
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, mes_data: int) -> DataFrame:
        self.logger.info(f"Construyendo antigüedad SUNAT con mes_data={mes_data}")
        df = self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmccapym_modelogestion_vu.ha_contribuyentesunatpyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            F.col("CTDMESANTIGUEDADEMP").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
        ).withColumn("codmes", add_codmes_spark("CODMES_0", +1)).drop("CODMES_0")
        return df

#utils/data_preparation/additional_variables_builder.py

In [0]:
# additional_variables_builder.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger

class AdditionalVariablesBuilder:
    """
    Carga tablas adicionales (VIDEA, matrices, movimientos, etc.) y realiza los joins
    con la base actual usando las claves codclaveunicocli y codinternocomputacional.
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def _get_joins_codclave(self, mes_actual_proceso: int):
        # Lista de DataFrames que se unen por codclaveunicocli
        # Definición simplificada; se pueden agregar todas las tablas del notebook.
        return [
            self._build_mtx_rcc_otra(mes_actual_proceso),
            self._build_clasi_exper(mes_actual_proceso),
            self._build_mtx_resumen_saldo(mes_actual_proceso),
            self._build_mtx_resumen_act_pas(mes_actual_proceso),
            # ... añadir el resto
        ]

    def _get_joins_codinterno(self, mes_actual_proceso: int):
        # Tablas que se unen por codinternocomputacional
        return [
            self._build_pasivo_evol(mes_actual_proceso),
            self._build_evol_cli(mes_actual_proceso)
        ]

    def _build_mtx_rcc_otra(self, mes):
        return self.spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccotradeuda"
        ).filter(F.col("codmes") == mes).select(
            "codmes", "codclaveunicocli",
            F.col("rcc_mto_rdv_max_u3m").alias("MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m")
        ).distinct()

    # ... implementar otros builders similares

    def join_all(self, base_df: DataFrame, mes_actual_proceso: int) -> DataFrame:
        self.logger.info("Uniendo tablas adicionales")
        # Joins por codclaveunicocli
        for df in self._get_joins_codclave(mes_actual_proceso):
            base_df = base_df.join(df, ["codmes", "codclaveunicocli"], "left")
        # Joins por codinternocomputacional
        for df in self._get_joins_codinterno(mes_actual_proceso):
            base_df = base_df.join(df, ["codmes", "codinternocomputacional"], "left")
        return base_df

#utils/data_preparation/modulo_activo_builder.py

In [0]:
# modulo_activo_builder.py
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType, DecimalType
from dateutil.relativedelta import relativedelta
from datetime import datetime
from utils.logger import MLOpsLogger

class ModuloActivoBuilder:
    """
    Construye variables del módulo activo: ratios de transacciones, variación de deuda,
    aplicando segmento de materialidad (flagctmay10omay6u12 = 1).
    """
    def __init__(self, spark, logger: MLOpsLogger):
        self.spark = spark
        self.logger = logger

    def build(self, universo_df, mes_actual_proceso: int, mes_data: int):
        self.logger.info("Construyendo variables del módulo activo")
        # Lógica completa del paso 10 del notebook
        # (se omite por brevedad, pero debe replicar el código original)
        # Retorna DataFrame con tres columnas: 
        # MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12,
        # MOD_ACT__pctratiomtodecdeudapymer tmtopasivoprmu3m,
        # MOD_ACT__pctratiomtoopeaprmu6mopecprmu12
        # ...

#utils/data_preparation/data_cleaner.py

In [0]:
# data_cleaner.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger

class DataCleaner:
    """
    Limpieza y transformaciones:
    - Reemplazo de valores dummy por NULL
    - Creación de variable RNG_ACTIVIDAD_ECONOM
    - Capping (winsorización) de columnas según umbrales predefinidos
    """
    def __init__(self, spark, logger: MLOpsLogger, caps_config: dict = None):
        self.spark = spark
        self.logger = logger
        self.caps_config = caps_config or {}  # diccionario col -> (lower, upper)

    def replace_dummy_values(self, df: DataFrame, dummy_list: list) -> DataFrame:
        self.logger.info("Reemplazando valores dummy por NULL")
        for col in df.columns:
            df = df.withColumn(col, F.when(F.col(col).isin(dummy_list), None).otherwise(F.col(col)))
        return df

    def add_activity_ring(self, df: DataFrame) -> DataFrame:
        self.logger.info("Creando variable RNG_ACTIVIDAD_ECONOM")
        actividades = ['PESCA', 'OTROS', 'SERVICIOS', 'ENERGIA', 'CONSTRUCCION', 'ADM_PUBLICA',
                       'ACT_INMOB', 'EMP_Y_DE_ALQ', 'INDUST_MANUFACT', 'COMERCIO', 'HOGAR', 'SALUD']
        return df.withColumn(
            "RNG_ACTIVIDAD_ECONOM",
            F.when(F.col("APP_SCORE_APROB_PYME__act_eco_fin").isNull(), 1)
             .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin").isin(actividades), 1)
             .otherwise(0)
        )

    def apply_capping(self, df: DataFrame) -> DataFrame:
        self.logger.info("Aplicando capping a variables")
        for col, (low, up) in self.caps_config.items():
            expr = F.col(col)
            if low is not None:
                expr = F.greatest(expr, F.lit(float(low)))
            if up is not None:
                expr = F.least(expr, F.lit(float(up)))
            df = df.withColumn(col + "_cap", expr)
        return df

    def run(self, df: DataFrame, dummy_list: list) -> DataFrame:
        df = self.replace_dummy_values(df, dummy_list)
        df = self.add_activity_ring(df)
        df = self.apply_capping(df)
        return df

#utils/data_preparation/feature_selector.py

In [0]:
# feature_selector.py
from pyspark.sql import DataFrame, functions as F
from utils.logger import MLOpsLogger

class FeatureSelector:
    """Selecciona las columnas finales según FEATURE_NAMES y las convierte a double."""
    def __init__(self, logger: MLOpsLogger):
        self.logger = logger

    def select(self, df: DataFrame, feature_names: list) -> DataFrame:
        self.logger.info(f"Seleccionando {len(feature_names)} features finales")
        return df.select(
            "codmes", "codclaveunicocli", "codclavepartycli", "codinternocomputacional",
            *[F.col(f).cast("double").alias(f) for f in feature_names]
        ).distinct()

#utils/data_preparation/table_writer.py

In [0]:
# table_writer.py
from pyspark.sql import DataFrame
from utils.logger import MLOpsLogger

class TableWriter:
    """Escribe el DataFrame final en una tabla Delta particionada por codmes."""
    def __init__(self, logger: MLOpsLogger):
        self.logger = logger

    def write(self, df: DataFrame, table_name: str, mode: str = "overwrite"):
        """
        Escribe la tabla con partición por codmes.
        mode: 'overwrite' o 'append'
        """
        self.logger.info(f"Escribiendo tabla {table_name} en modo {mode}")
        df.write.mode(mode) \
            .option("overwriteSchema", "true") \
            .partitionBy("codmes") \
            .saveAsTable(table_name)

#utils/data_preparation/pipeline.py (orquestador)

In [0]:
# pipeline.py
from pyspark.sql import SparkSession
from utils.logger import MLOpsLogger
from utils.data_preparation import (
    UniverseBuilder, RelationshipBuilder, DemographicBuilder, CarreteraBuilder,
    OwnerVariablesBuilder, AggregatedVariablesBuilder, SunatSeniorityBuilder,
    AdditionalVariablesBuilder, ModuloActivoBuilder, DataCleaner, FeatureSelector, TableWriter
)
from utils.month_delta import month_delta

class DataPreparationPipeline:
    """
    Orquestador principal: ejecuta todos los pasos en el orden correcto,
    maneja el modo histórico y la escritura con sufijo de versión.
    """
    def __init__(self, spark: SparkSession, logger: MLOpsLogger, config: dict):
        self.spark = spark
        self.logger = logger
        self.config = config
        # Instanciar builders (se pueden pasar dependencias)
        self.universe_builder = UniverseBuilder(spark, logger)
        self.relationship_builder = RelationshipBuilder(spark, logger)
        self.demographic_builder = DemographicBuilder(spark, logger)
        self.carretera_builder = CarreteraBuilder(spark, logger)
        self.owner_vars_builder = OwnerVariablesBuilder(spark, logger)
        self.ag_vars_builder = AggregatedVariablesBuilder(spark, logger)
        self.sunat_builder = SunatSeniorityBuilder(spark, logger)
        self.additional_builder = AdditionalVariablesBuilder(spark, logger)
        self.modulo_activo_builder = ModuloActivoBuilder(spark, logger)
        self.data_cleaner = DataCleaner(spark, logger, caps_config={})  # cargar umbrales
        self.feature_selector = FeatureSelector(logger)
        self.table_writer = TableWriter(logger)

    def run(self, meses_a_procesar: list, output_table_base: str, suffix: str = ""):
        """
        meses_a_procesar: lista de meses (enteros) a procesar (p.ej. [202601])
        output_table_base: nombre base de la tabla (sin sufijo)
        suffix: sufijo a añadir (ej. "_v2") para no sobrescribir versiones anteriores
        """
        output_table = f"{output_table_base}{suffix}"
        for idx, mes in enumerate(meses_a_procesar):
            mes_data = int(month_delta(str(mes), -1))
            write_mode = "overwrite" if idx == 0 else "append"

            # 1. Universo
            universo = self.universe_builder.build(self.config["SOURCE_TABLE"], mes)
            # 2. Relaciones
            relacion = self.relationship_builder.build(mes_data)
            # 3. Demográficas
            demo = self.demographic_builder.build(mes_data)
            # 4. Carretera
            carretera = self.carretera_builder.build(mes_data)
            # 5. Variables dueño
            owner_vars = self.owner_vars_builder.build(relacion, carretera)
            # 6. Variables agregadas
            base = self.ag_vars_builder.build(universo, demo, owner_vars, carretera)
            # 7. Antigüedad SUNAT
            sunat = self.sunat_builder.build(mes_data)
            base = base.join(sunat, ["codclaveunicocli", "codmes"], "left_outer")
            # 8. Tablas adicionales
            base = self.additional_builder.join_all(base, mes)
            # 9. Módulo activo
            mod_act = self.modulo_activo_builder.build(universo, mes, mes_data)
            base = base.join(mod_act, ["codmes", "codclaveunicocli"], "left_outer")
            # 10. Limpieza (dummy, actividad, capping)
            dummy_list = [1111111111, -1111111111, ...]  # desde config
            base_clean = self.data_cleaner.run(base, dummy_list)
            # 11. Seleccionar features
            features_df = self.feature_selector.select(base_clean, self.config["FEATURE_NAMES"])
            # 12. Escribir tabla
            self.table_writer.write(features_df, output_table, mode=write_mode)
        return output_table